# 🧪 TestMate — Functional Evaluation on V2 Adapter



## 1️⃣ Install Dependencies

In [1]:
!pip uninstall -y transformers huggingface_hub accelerate peft bitsandbytes
!pip install -q \
    transformers==4.38.2 \
    huggingface_hub==0.21.4 \
    accelerate==0.27.2 \
    peft==0.8.2 \
    bitsandbytes==0.42.0

import os, torch
os.environ['LD_LIBRARY_PATH'] = '/usr/local/cuda/lib64'
!pip install -qU bitsandbytes --no-cache-dir

print(f'✅ PyTorch : {torch.__version__}')
print(f'✅ CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'✅ GPU     : {torch.cuda.get_device_name(0)}')

Found existing installation: transformers 5.2.0
Uninstalling transformers-5.2.0:
  Successfully uninstalled transformers-5.2.0
Found existing installation: huggingface_hub 1.4.1
Uninstalling huggingface_hub-1.4.1:
  Successfully uninstalled huggingface_hub-1.4.1
Found existing installation: accelerate 1.12.0
Uninstalling accelerate-1.12.0:
  Successfully uninstalled accelerate-1.12.0
Found existing installation: peft 0.18.1
Uninstalling peft-0.18.1:
  Successfully uninstalled peft-0.18.1
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.7/130.7 kB 3.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 74.9 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.4/346.4 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.0/280.0 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.0/105.0 MB 18.3 MB/s eta 0:00:0000:0100

## 2️⃣ Load V2 Adapter (no training)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import torch

BASE_MODEL   = "Qwen/Qwen2.5-Coder-7B-Instruct"
ADAPTER_PATH = "/kaggle/input/models/esraataha28/final/pytorch/default/1/Final" 
CACHE_DIR    = "/kaggle/working/model_cache"

# ── Load base model (4-bit) ──────────────────────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16
)

print('⏳ Loading base model...')
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    cache_dir=CACHE_DIR,
    trust_remote_code=True
)

# ── Load Tokenizer ───────────────────────────────────────────────
print('⏳ Loading Tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,        
    trust_remote_code=True,
    cache_dir=CACHE_DIR
)
tokenizer.pad_token = tokenizer.eos_token
# ── Merge LoRA adapter ───────────────────────────────────────────────
print('⏳ Loading V2 LoRA adapter...')
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()
print('✅ V2 Adapter loaded and ready!')

⏳ Loading base model...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

⏳ Loading Tokenizer...


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


⏳ Loading V2 LoRA adapter...
✅ V2 Adapter loaded and ready!


## 3️⃣ Functional Test Suite

In [7]:
# ── Helper classes ───────────────────────────────────────────────────
class Node:
    def __init__(self, name):
        self.name = name
        self.successors  = []
        self.predecessors = []
    def __repr__(self): return f'Node({self.name})'

class LinkedNode:
    def __init__(self, val):
        self.val  = val
        self.next = None

def make_list(*vals):
    head = LinkedNode(vals[0])
    cur  = head
    for v in vals[1:]:
        cur.next = LinkedNode(v)
        cur = cur.next
    return head

def make_cycle_list(*vals):
    head = make_list(*vals)
    cur  = head
    while cur.next: cur = cur.next
    cur.next = head   # cycle!
    return head

def make_graph_line(names):
    nodes = [Node(n) for n in names]
    for i in range(len(nodes) - 1):
        nodes[i].successors.append(nodes[i+1])
        nodes[i+1].predecessors.append(nodes[i])
    return nodes

print('✅ Helpers ready')

✅ Helpers ready


In [8]:
BENCHMARK = {
    'mergesort': {
        'buggy': """def merge(left, right):
    result = []
    i = j = 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]: result.append(left[i]); i += 1
        else: result.append(right[j]); j += 1
    result.extend(left[i:]); result.extend(right[j:])
    return result

def mergesort(arr):
    if len(arr) <= 1: return arr
    mid = len(arr) // 2
    left  = mergesort(arr[:mid])
    right = mergesort(arr[mid:])
    return list(merge(left, right))""",
        'issue': 'Fix mergesort to correctly split and merge.',
        'tests': [
            lambda ns: (ns['mergesort']([3,1,4,1,5]) == [1,1,3,4,5], 'mixed list'),
            lambda ns: (ns['mergesort']([])           == [],           'empty'),
            lambda ns: (ns['mergesort']([5,4,3,2,1]) == [1,2,3,4,5], 'reverse'),
        ]
    },
    'levenshtein': {
        'buggy': """def levenshtein(s, t):
    if not s or not t: return len(s) or len(t)
    elif s[0] == t[0]: return levenshtein(s[1:], t[1:])
    else: return 1 + min(levenshtein(s, t[1:]), levenshtein(s[1:], t), levenshtein(s, t[1:]))""",
        'issue': 'Fix levenshtein — third min arg must be levenshtein(s[1:], t[1:]).',
        'tests': [
            lambda ns: (ns['levenshtein']('kitten','sitting') == 3, 'kitten→sitting=3'),
            lambda ns: (ns['levenshtein']('','abc')           == 3, 'empty→abc=3'),
            lambda ns: (ns['levenshtein']('abc','abc')        == 0, 'identical=0'),
        ]
    },
    'lis': {
        'buggy': """def lis(arr):
    ends = {}; longest = 0
    for i, val in enumerate(arr):
        best_len = 0
        for length in range(1, longest + 1):
            if arr[ends[length - 1]] < val: best_len = length; break
        ends[best_len] = i
        longest = max(longest, best_len + 1)
    return longest""",
        'issue': 'Fix LIS — use list comprehension to find max prefix length.',
        'tests': [
            lambda ns: (ns['lis']([4,1,5,2,3]) == 3, '[4,1,5,2,3]→3'),
            lambda ns: (ns['lis']([1,2,3,4,5]) == 5, 'sorted→5'),
            lambda ns: (ns['lis']([5,4,3,2,1]) == 1, 'descending→1'),
        ]
    },
    'knapsack': {
        'buggy': """def knapsack(capacity, items):
    memo = {}
    for i in range(len(items)+1):
        for j in range(capacity+1):
            if i==0 or j==0: memo[i,j]=0
            elif items[i-1][1]<=j:
                memo[i,j]=max(memo[i-1,j], memo[i-1,j-items[i-1][1]]+items[i-1][1])
            else: memo[i,j]=memo[i-1,j]
    return memo[len(items),capacity]""",
        'issue': 'Fix knapsack — use items[i-1][0] for weight, items[i-1][1] for value.',
        'tests': [
            lambda ns: (ns['knapsack'](50,[(10,60),(20,100),(30,120)]) == 220, '3-item classic=220'),
            lambda ns: (ns['knapsack'](0, [(1,10)])                   == 0,   'zero capacity=0'),
            lambda ns: (ns['knapsack'](10,[])                          == 0,   'no items=0'),
        ]
    },
    'detect_cycle': {
        'buggy': """def detect_cycle(node):
    hare = tortoise = node
    while hare.next:
        tortoise = tortoise.next
        hare = hare.next.next
        if hare == tortoise: return True
    return False""",
        'issue': 'Fix detect_cycle — guard: while hare and hare.next.',
        'tests': [
            lambda ns: (ns['detect_cycle'](make_cycle_list(1,2,3,4)) == True,  'cycle→True'),
            lambda ns: (ns['detect_cycle'](make_list(1,2,3,4))       == False, 'no cycle→False'),
            lambda ns: (ns['detect_cycle'](make_list(1))             == False, 'single→False'),
        ]
    },
    'sqrt': {
        'buggy': """def sqrt(x, epsilon):
    approx = x / 2
    while abs(approx**2 - x) > epsilon:
        approx = 0.5 * (approx + x / approx)
    return approx""",
        'issue': 'Fix sqrt — abs order should be abs(x - approx**2).',
        'tests': [
            lambda ns: (abs(ns['sqrt'](4, 0.0001) - 2.0)   < 0.001, 'sqrt(4)≈2'),
            lambda ns: (abs(ns['sqrt'](9, 0.0001) - 3.0)   < 0.001, 'sqrt(9)≈3'),
            lambda ns: (abs(ns['sqrt'](2, 0.0001) - 1.414) < 0.001, 'sqrt(2)≈1.414'),
        ]
    },
    'powerset': {
        'buggy': """def powerset(arr):
    if not arr: return [[]]
    first = arr[0]; rest = powerset(arr[1:])
    return [[first]+s for s in rest]""",
        'issue': 'Fix powerset — return rest + [[first]+s for s in rest].',
        'tests': [
            lambda ns: (len(ns['powerset']([1,2,3]))   == 8,  '[1,2,3]→8 subsets'),
            lambda ns: (len(ns['powerset']([1,2,3,4])) == 16, '[1,2,3,4]→16 subsets'),
            lambda ns: (ns['powerset']([])             == [[]],'[]→[[]]'),
        ]
    },
    'breadth_first_search': {
        'buggy': """def breadth_first_search(startnode, goalnode):
    queue = [startnode]; visited = set()
    while queue:
        node = queue.pop(0)
        if node is goalnode: return True
        for s in node.successors:
            if s not in visited: visited.add(s); queue.append(s)
    return False""",
        'issue': 'Fix BFS — initialize visited = {startnode}.',
        'tests': [],
        'multi_test': lambda ns: [
            (ns['breadth_first_search'](*make_graph_line(['A','B','C','D'])[::3]) == True,  'A→D reachable'),
            (ns['breadth_first_search'](Node('X'), Node('Y'))                    == False, 'disconnected'),
            (lambda n: ns['breadth_first_search'](n, n))(make_graph_line(['A'])[0]) == True
            and (True, 'start==goal') or (False, 'start==goal'),
        ]
    },
    'depth_first_search': {
        'buggy': """def depth_first_search(startnode, goalnode):
    visited = set()
    def search(node):
        if node in visited: return False
        visited.add(node)
        if node is goalnode: return True
        for s in node.successors:
            if search(s): return True
        return False
    return search(startnode)""",
        'issue': 'Fix DFS — check goalnode first, then mark visited.',
        'tests': [],
        'multi_test': lambda ns: [
            (ns['depth_first_search'](*make_graph_line(['A','B','C','D'])[::3]) == True,  'A→D reachable'),
            (ns['depth_first_search'](Node('X'), Node('Y'))                    == False, 'disconnected'),
        ]
    },
    'sieve': {
        'buggy': """def sieve(max_val):
    primes = []
    for n in range(2, max_val+1):
        if all(n % p == 0 for p in primes): primes.append(n)
    return primes""",
        'issue': 'Fix sieve — condition must be n % p > 0, not == 0.',
        'tests': [
            lambda ns: (ns['sieve'](10) == [2,3,5,7],              'primes≤10'),
            lambda ns: (ns['sieve'](20) == [2,3,5,7,11,13,17,19], 'primes≤20'),
            lambda ns: (ns['sieve'](1)  == [],                     'sieve(1)=[]'),
        ]
    },
}
print(f'✅ {len(BENCHMARK)} benchmark algorithms loaded')

✅ 10 benchmark algorithms loaded


## 4️⃣ Run Functional Benchmark

In [9]:
import re, textwrap
from tqdm import tqdm

def generate_fix(buggy: str, issue: str, max_new_tokens=512) -> str:
    prompt = (
        "<|im_start|>system\n"
        "You are an expert APR agent. Return ONLY the corrected function(s), no explanation.<|im_end|>\n"
        "<|im_start|>user\n"
        f"ISSUE: {issue}\n\nBUGGY CODE:\n{buggy}<|im_end|>\n"
        "<|im_start|>assistant\n"
    )
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    gen = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return re.sub(r'```[a-z]*\n?', '', gen).strip()

def run_tests(code_str, test_fns, multi_test=None):
    ns = {'make_list': make_list, 'make_cycle_list': make_cycle_list,
          'make_graph_line': make_graph_line, 'Node': Node}
    try:
        exec(textwrap.dedent(code_str), ns)
    except Exception as e:
        return 0, 1, [f'  💥 EXEC ERROR: {e}']

    results, passed = [], 0
    for fn in test_fns:
        try:
            ok, desc = fn(ns)
            results.append(f'  {"✅" if ok else "❌"} {desc}')
            if ok: passed += 1
        except Exception as e:
            results.append(f'  💥 {e}')

    if multi_test:
        try:
            for item in multi_test(ns):
                if isinstance(item, tuple) and len(item) == 2:
                    ok, desc = item
                    results.append(f'  {"✅" if ok else "❌"} {desc}')
                    if ok: passed += 1
        except Exception as e:
            results.append(f'  💥 multi_test: {e}')

    return passed, len(results), results

# ── Run ──────────────────────────────────────────────────────────────
scores = {'pass': 0, 'partial': 0, 'fail': 0}

print('🏆 Functional Benchmark — V2 Adapter\n')
print('=' * 60)

for name, data in tqdm(BENCHMARK.items()):
    print(f'\n📌 {name.upper()}')
    gen = generate_fix(data['buggy'], data['issue'])
    print(f'\n🤖 Fix:\n{gen}\n')

    passed, total, details = run_tests(
        gen, data.get('tests', []), data.get('multi_test')
    )
    for d in details: print(d)

    ratio = passed / total if total > 0 else 0
    if   ratio == 1.0: label = f'✨ PASS    ({passed}/{total})'; scores['pass']    += 1
    elif ratio  > 0:   label = f'⚠️  PARTIAL ({passed}/{total})'; scores['partial'] += 1
    else:              label = f'❌ FAIL    ({passed}/{total})'; scores['fail']    += 1
    print(f'\nResult: {label}')
    print('-' * 60)

functional = scores['pass'] + 0.5 * scores['partial']
print(f'\n🏆 Final Results (V2 Adapter, Functional Eval):')
print(f'   ✨ PASS    : {scores["pass"]}/10')
print(f'   ⚠️  PARTIAL : {scores["partial"]}/10')
print(f'   ❌ FAIL    : {scores["fail"]}/10')
print(f'   📊 Score   : {functional:.1f} / 10')

🏆 Functional Benchmark — V2 Adapter



  0%|          | 0/10 [00:00<?, ?it/s]


📌 MERGESORT


2026-02-24 19:20:20.233259: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771960820.466509      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771960820.533872      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771960821.072994      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771960821.073029      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771960821.073032      55 computation_placer.cc:177] computation placer alr


🤖 Fix:
def merge(left, right):
    result = []
    i = j = 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]: result.append(left[i]); i += 1
        else: result.append(right[j]); j += 1
    result.extend(left[i:]); result.extend(right[j:])
    return result

def mergesort(arr):
    if len(arr) <= 1: return arr
    mid = len(arr) // 2
    left  = mergesort(arr[:mid])
    right = mergesort(arr[mid:])
    return merge(left, right)

  ✅ mixed list
  ✅ empty
  ✅ reverse

Result: ✨ PASS    (3/3)
------------------------------------------------------------

📌 LEVENSHTEIN


...


🤖 Fix:
def breadth_first_search(startnode, goalnode):
    queue = [startnode]; visited = {startnode}
    while queue:
        node = queue.pop(0)
        if node is goalnode: return True
        for s in node.successors:
            if s not in visited: visited.add(s); queue.append(s)
    return False

  ✅ A→D reachable
  ✅ disconnected
  ✅ start==goal

Result: ✨ PASS    (3/3)
------------------------------------------------------------

📌 DEPTH_FIRST_SEARCH


 90%|█████████ | 9/10 [02:04<00:09,  9.90s/it]


🤖 Fix:
def depth_first_search(startnode, goalnode):
    visited = set()
    def search(node):
        if node is goalnode: return True
        if node in visited: return False
        visited.add(node)
        for s in node.successors:
            if search(s): return True
        return False
    return search(startnode)

  ✅ A→D reachable
  ✅ disconnected

Result: ✨ PASS    (2/2)
------------------------------------------------------------

📌 SIEVE


100%|██████████| 10/10 [02:10<00:00, 13.07s/it]


🤖 Fix:
def sieve(max_val):
    primes = []
    for n in range(2, max_val+1):
        if all(n % p > 0 for p in primes): primes.append(n)
    return primes

  ✅ primes≤10
  ✅ primes≤20
  ✅ sieve(1)=[]

Result: ✨ PASS    (3/3)
------------------------------------------------------------

🏆 Final Results (V2 Adapter, Functional Eval):
   ✨ PASS    : 10/10
   ⚠️  PARTIAL : 0/10
   ❌ FAIL    : 0/10
   📊 Score   : 10.0 / 10


In [2]:
import os, torch, re, textwrap
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from tqdm import tqdm

BASE_MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"
CACHE_DIR  = "/kaggle/working/model_cache"

print('⏳ Loading BASE model (No Training)...')
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True, cache_dir=CACHE_DIR)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    cache_dir=CACHE_DIR,
    trust_remote_code=True
)
model.eval()
print('✅ Base Model loaded and ready!')

class Node:
    def __init__(self, name):
        self.name = name
        self.successors  = []
        self.predecessors = []
    def __repr__(self): return f'Node({self.name})'

class LinkedNode:
    def __init__(self, val):
        self.val  = val
        self.next = None

def make_list(*vals):
    head = LinkedNode(vals[0]); cur = head
    for v in vals[1:]: cur.next = LinkedNode(v); cur = cur.next
    return head

def make_cycle_list(*vals):
    head = make_list(*vals); cur = head
    while cur.next: cur = cur.next
    cur.next = head
    return head

def make_graph_line(names):
    nodes = [Node(n) for n in names]
    for i in range(len(nodes) - 1):
        nodes[i].successors.append(nodes[i+1])
        nodes[i+1].predecessors.append(nodes[i])
    return nodes

BENCHMARK = {
    'mergesort': {
        'buggy': "def merge(left, right):\n    result = []\n    i = j = 0\n    while i < len(left) and j < len(right):\n        if left[i] <= right[j]: result.append(left[i]); i += 1\n        else: result.append(right[j]); j += 1\n    result.extend(left[i:]); result.extend(right[j:])\n    return result\n\ndef mergesort(arr):\n    if len(arr) <= 1: return arr\n    mid = len(arr) // 2\n    left  = mergesort(arr[:mid])\n    right = mergesort(arr[mid:])\n    return list(merge(left, right))",
        'issue': 'Fix mergesort to correctly split and merge.',
        'tests': [
            lambda ns: (ns['mergesort']([3,1,4,1,5]) == [1,1,3,4,5], 'mixed list'),
            lambda ns: (ns['mergesort']([])           == [],           'empty'),
            lambda ns: (ns['mergesort']([5,4,3,2,1]) == [1,2,3,4,5], 'reverse'),
        ]
    },
    'levenshtein': {
        'buggy': "def levenshtein(s, t):\n    if not s or not t: return len(s) or len(t)\n    elif s[0] == t[0]: return levenshtein(s[1:], t[1:])\n    else: return 1 + min(levenshtein(s, t[1:]), levenshtein(s[1:], t), levenshtein(s, t[1:]))",
        'issue': 'Fix levenshtein — third min arg must be levenshtein(s[1:], t[1:]).',
        'tests': [
            lambda ns: (ns['levenshtein']('kitten','sitting') == 3, 'kitten→sitting=3'),
            lambda ns: (ns['levenshtein']('','abc')           == 3, 'empty→abc=3'),
            lambda ns: (ns['levenshtein']('abc','abc')        == 0, 'identical=0'),
        ]
    },
    'lis': {
        'buggy': "def lis(arr):\n    ends = {}; longest = 0\n    for i, val in enumerate(arr):\n        best_len = 0\n        for length in range(1, longest + 1):\n            if arr[ends[length - 1]] < val: best_len = length; break\n        ends[best_len] = i\n        longest = max(longest, best_len + 1)\n    return longest",
        'issue': 'Fix LIS — use list comprehension to find max prefix length.',
        'tests': [
            lambda ns: (ns['lis']([4,1,5,2,3]) == 3, '[4,1,5,2,3]→3'),
            lambda ns: (ns['lis']([1,2,3,4,5]) == 5, 'sorted→5'),
            lambda ns: (ns['lis']([5,4,3,2,1]) == 1, 'descending→1'),
        ]
    },
    'knapsack': {
        'buggy': "def knapsack(capacity, items):\n    memo = {}\n    for i in range(len(items)+1):\n        for j in range(capacity+1):\n            if i==0 or j==0: memo[i,j]=0\n            elif items[i-1][1]<=j:\n                memo[i,j]=max(memo[i-1,j], memo[i-1,j-items[i-1][1]]+items[i-1][1])\n            else: memo[i,j]=memo[i-1,j]\n    return memo[len(items),capacity]",
        'issue': 'Fix knapsack — use items[i-1][0] for weight, items[i-1][1] for value.',
        'tests': [
            lambda ns: (ns['knapsack'](50,[(10,60),(20,100),(30,120)]) == 220, '3-item classic=220'),
            lambda ns: (ns['knapsack'](0, [(1,10)])                   == 0,   'zero capacity=0'),
            lambda ns: (ns['knapsack'](10,[])                          == 0,   'no items=0'),
        ]
    },
    'detect_cycle': {
        'buggy': "def detect_cycle(node):\n    hare = tortoise = node\n    while hare.next:\n        tortoise = tortoise.next\n        hare = hare.next.next\n        if hare == tortoise: return True\n    return False",
        'issue': 'Fix detect_cycle — guard: while hare and hare.next.',
        'tests': [
            lambda ns: (ns['detect_cycle'](make_cycle_list(1,2,3,4)) == True,  'cycle→True'),
            lambda ns: (ns['detect_cycle'](make_list(1,2,3,4))       == False, 'no cycle→False'),
            lambda ns: (ns['detect_cycle'](make_list(1))             == False, 'single→False'),
        ]
    },
    'sqrt': {
        'buggy': "def sqrt(x, epsilon):\n    approx = x / 2\n    while abs(approx**2 - x) > epsilon:\n        approx = 0.5 * (approx + x / approx)\n    return approx",
        'issue': 'Fix sqrt — abs order should be abs(x - approx**2).',
        'tests': [
            lambda ns: (abs(ns['sqrt'](4, 0.0001) - 2.0)   < 0.001, 'sqrt(4)≈2'),
            lambda ns: (abs(ns['sqrt'](9, 0.0001) - 3.0)   < 0.001, 'sqrt(9)≈3'),
            lambda ns: (abs(ns['sqrt'](2, 0.0001) - 1.414) < 0.001, 'sqrt(2)≈1.414'),
        ]
    },
    'powerset': {
        'buggy': "def powerset(arr):\n    if not arr: return [[]]\n    first = arr[0]; rest = powerset(arr[1:])\n    return [[first]+s for s in rest]",
        'issue': 'Fix powerset — return rest + [[first]+s for s in rest].',
        'tests': [
            lambda ns: (len(ns['powerset']([1,2,3]))   == 8,  '[1,2,3]→8 subsets'),
            lambda ns: (len(ns['powerset']([1,2,3,4])) == 16, '[1,2,3,4]→16 subsets'),
            lambda ns: (ns['powerset']([])             == [[]],'[]→[[]]'),
        ]
    },
    'breadth_first_search': {
        'buggy': "def breadth_first_search(startnode, goalnode):\n    queue = [startnode]; visited = set()\n    while queue:\n        node = queue.pop(0)\n        if node is goalnode: return True\n        for s in node.successors:\n            if s not in visited: visited.add(s); queue.append(s)\n    return False",
        'issue': 'Fix BFS — initialize visited = {startnode}.',
        'tests': [],
        'multi_test': lambda ns: [
            (ns['breadth_first_search'](*make_graph_line(['A','B','C','D'])[::3]) == True,  'A→D reachable'),
            (ns['breadth_first_search'](Node('X'), Node('Y'))                    == False, 'disconnected'),
            (lambda n: ns['breadth_first_search'](n, n))(make_graph_line(['A'])[0]) == True and (True, 'start==goal') or (False, 'start==goal'),
        ]
    },
    'depth_first_search': {
        'buggy': "def depth_first_search(startnode, goalnode):\n    visited = set()\n    def search(node):\n        if node in visited: return False\n        visited.add(node)\n        if node is goalnode: return True\n        for s in node.successors:\n            if search(s): return True\n        return False\n    return search(startnode)",
        'issue': 'Fix DFS — check goalnode first, then mark visited.',
        'tests': [],
        'multi_test': lambda ns: [
            (ns['depth_first_search'](*make_graph_line(['A','B','C','D'])[::3]) == True,  'A→D reachable'),
            (ns['depth_first_search'](Node('X'), Node('Y'))                    == False, 'disconnected'),
        ]
    },
    'sieve': {
        'buggy': "def sieve(max_val):\n    primes = []\n    for n in range(2, max_val+1):\n        if all(n % p == 0 for p in primes): primes.append(n)\n    return primes",
        'issue': 'Fix sieve — condition must be n % p > 0, not == 0.',
        'tests': [
            lambda ns: (ns['sieve'](10) == [2,3,5,7],              'primes≤10'),
            lambda ns: (ns['sieve'](20) == [2,3,5,7,11,13,17,19], 'primes≤20'),
            lambda ns: (ns['sieve'](1)  == [],                     'sieve(1)=[]'),
        ]
    },
}

def generate_fix(buggy: str, issue: str, max_new_tokens=512) -> str:
    prompt = (
        "<|im_start|>system\n"
        "You are an expert APR agent. Return ONLY the corrected function(s), no explanation.<|im_end|>\n"
        "<|im_start|>user\n"
        f"ISSUE: {issue}\n\nBUGGY CODE:\n{buggy}<|im_end|>\n"
        "<|im_start|>assistant\n"
    )
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    gen = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return re.sub(r'```[a-z]*\n?', '', gen).strip()

def run_tests(code_str, test_fns, multi_test=None):
    ns = {'make_list': make_list, 'make_cycle_list': make_cycle_list, 'make_graph_line': make_graph_line, 'Node': Node}
    try:
        exec(textwrap.dedent(code_str), ns)
    except Exception as e:
        return 0, 1, [f'  💥 EXEC ERROR: {e}']

    results, passed = [], 0
    for fn in test_fns:
        try:
            ok, desc = fn(ns)
            results.append(f'  {"✅" if ok else "❌"} {desc}')
            if ok: passed += 1
        except Exception as e:
            results.append(f'  💥 {e}')

    if multi_test:
        try:
            for item in multi_test(ns):
                if isinstance(item, tuple) and len(item) == 2:
                    ok, desc = item
                    results.append(f'  {"✅" if ok else "❌"} {desc}')
                    if ok: passed += 1
        except Exception as e:
            results.append(f'  💥 multi_test: {e}')

    return passed, max(len(results), 1), results

scores = {'pass': 0, 'fail': 0}

print('🏆 Functional Benchmark — BASE MODEL (No Training)\n')
print('=' * 60)

for name, data in tqdm(BENCHMARK.items()):
    print(f'\n📌 {name.upper()}')
    gen = generate_fix(data['buggy'], data['issue'])
    print(f'\n🤖 Fix:\n{gen}\n')

    passed, total, details = run_tests(gen, data.get('tests', []), data.get('multi_test'))
    for d in details: print(d)

    ratio = passed / total if total > 0 else 0
    
    # Strict Evaluation: Only PASS if 100% of tests are successful
    if ratio == 1.0: 
        label = f'✨ PASS    ({passed}/{total})'
        scores['pass'] += 1
    else: 
        label = f'❌ FAIL    ({passed}/{total})'
        scores['fail'] += 1
        
    print(f'\nResult: {label}')
    print('-' * 60)

print(f'\n🏆 Final Results (Base Model - Un-trained):')
print(f'   ✨ PASS    : {scores["pass"]}/10')
print(f'   ❌ FAIL    : {scores["fail"]}/10')
print(f'   📊 Score   : {scores["pass"]:.1f} / 10')

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

⏳ Loading BASE model (No Training)...


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

...


🤖 Fix:
def breadth_first_search(startnode, goalnode):
    queue = [startnode]
    visited = {startnode}
    while queue:
        node = queue.pop(0)
        if node == goalnode: return True
        for s in node.successors:
            if s not in visited:
                visited.add(s)
                queue.append(s)
    return False

  ✅ A→D reachable
  ✅ disconnected
  ✅ start==goal

Result: ✨ PASS    (3/3)
------------------------------------------------------------

📌 DEPTH_FIRST_SEARCH


 90%|█████████ | 9/10 [01:26<00:06,  6.12s/it]


🤖 Fix:
def depth_first_search(startnode, goalnode):
    visited = set()
    def search(node):
        if node is goalnode: return True
        if node in visited: return False
        visited.add(node)
        for s in node.successors:
            if search(s): return True
        return False
    return search(startnode)

  ✅ A→D reachable
  ✅ disconnected

Result: ✨ PASS    (2/2)
------------------------------------------------------------

📌 SIEVE


100%|██████████| 10/10 [01:30<00:00,  9.02s/it]


🤖 Fix:
def sieve(max_val):
    primes = []
    for n in range(2, max_val+1):
        if all(n % p != 0 for p in primes): primes.append(n)
    return primes

  ✅ primes≤10
  ✅ primes≤20
  ✅ sieve(1)=[]

Result: ✨ PASS    (3/3)
------------------------------------------------------------

🏆 Final Results (Base Model - Un-trained):
   ✨ PASS    : 8/10
   ❌ FAIL    : 2/10
   📊 Score   : 8.0 / 10
